# ORB Final Candidate Notebook

Notebook de validation visuelle et quantitative du candidat final de campagne: **`b_atr_q20_q90`** (filtre ATR born? au moment du signal).

Objectif: comparer proprement **baseline vs candidat** sur le m?me pipeline et v?rifier la coh?rence avec les exports de campagne.


In [1]:
from pathlib import Path
import sys

root = Path.cwd().resolve()
while root != root.parent and not (root / 'pyproject.toml').exists():
    root = root.parent

if not (root / 'pyproject.toml').exists():
    raise RuntimeError('Repository root not found from current working directory.')

if str(root) not in sys.path:
    sys.path.insert(0, str(root))

print(f'Project root: {root}')


Project root: D:\Business\Trading\VSCODE\algo-trading-intraday-research


In [2]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src.analytics.metrics import compute_metrics
from src.config.settings import DEFAULT_INITIAL_CAPITAL_USD
from src.data.cleaning import clean_ohlcv
from src.data.loader import load_ohlcv_file
from src.engine.backtester import run_backtest
from src.engine.execution_model import ExecutionModel
from src.engine.portfolio import build_equity_curve
from src.features.intraday import add_intraday_features, add_session_vwap, add_continuous_session_vwap
from src.features.opening_range import compute_opening_range
from src.features.volatility import add_atr
from src.strategy.orb import ORBStrategy

pd.set_option('display.width', 220)
pd.set_option('display.max_columns', 120)
plt.style.use('seaborn-v0_8-whitegrid')


In [3]:
RUN_DIR = root / 'data' / 'exports' / 'orb_robust_campaign_20260320_165646'
DATASET_PATH = root / 'data' / 'raw' / 'NQ_1min.parquet'

variants = pd.read_csv(RUN_DIR / 'campaigns' / 'variants_consolidated.csv')
reco = pd.read_csv(RUN_DIR / 'summary' / 'final_recommendation_table.csv')
finalists_overall = pd.read_csv(RUN_DIR / 'finalists' / 'finalists_overall_metrics.csv')

candidate_name = str(reco.iloc[0]['candidate'])
candidate_row = variants.loc[variants['name'] == candidate_name].iloc[0]

display(reco)
display(candidate_row.to_frame().T)


,candidate,improves_overall_metric_count,oos_check_passed,year_stability_passed,beats_baseline,overall_sharpe,overall_profit_factor,overall_expectancy,overall_abs_drawdown
0,b_atr_q20_q90,4,True,True,True,1.089292,1.294469,25.470812,2083.5
1,b_atr_ge_q30,4,True,True,True,0.802025,1.205877,19.207751,2298.5
2,a_ratio_q10_q90,4,True,True,True,0.764521,1.183978,16.807564,2080.0


,name,block,hypothesis,column,lower,upper,strictly_positive,candidate_signals_before_filter,candidate_signals_after_filter,selected_signal_days,n_trades,win_rate,avg_win,avg_loss,average_losing_trade,expectancy,profit_factor,cumulative_pnl,max_drawdown,max_drawdown_pct,sharpe_ratio,max_consecutive_losses,avg_consecutive_losses,longest_loss_streak,number_of_loss_streaks_2_plus,worst_trade,worst_day,stop_hit_rate,target_hit_rate,time_exit_rate,time_exit_count,time_exit_win_rate,time_exit_loss_rate,time_exit_pnl_usd,eod_exit_rate,proportion_filtered_out,trade_count_after_filters,percent_of_days_traded,percent_days_traded,avg_R,median_R,pnl_to_risk_ratio,average_loss_streak_length,count_loss_streaks_2_plus,daily_sharpe_basis,n_sessions,raw_signal_count,delta_sharpe_pct,delta_profit_factor_pct,delta_expectancy_pct,drawdown_improvement_pct,delta_win_rate_pct,trade_floor_ok,improvement_count,major_degrade,signal_credible_phase3,phase3_robustness_score
0,b_atr_q20_q90,B_atr_regime,Avoid low-vol and extreme-vol ATR tails at sig...,atr_14,6.839286,21.546429,False,1340,865,865,788,0.493655,226.81491,-170.827068,-170.827068,25.470812,1.294469,20071.0,-2083.5,0.04167,1.089292,8,1.946341,8,107,-250.0,-250.0,0.449239,0.211929,0.333756,263,0.828897,0.171103,25737.0,0.005076,0.645055,788,0.370475,0.370475,0.136417,-0.032043,0.136481,1.946341,107,daily_pnl_over_static_capital,2127,2437,57.884998,12.792943,90.156372,27.645788,7.006103,True,4,False,True,0.461809


In [4]:
baseline_cfg = {
    'or_minutes': 15,
    'opening_time': '09:30:00',
    'direction': 'long',
    'entry_buffer_ticks': 2,
    'stop_buffer_ticks': 2,
    'target_multiple': 2.0,
    'time_exit': '16:00:00',
    'vwap_column': 'continuous_session_vwap',
    'account_size_usd': DEFAULT_INITIAL_CAPITAL_USD,
    'risk_per_trade_pct': 0.5,
}

# Full-session feature flow identical to campaign runner.
df = load_ohlcv_file(DATASET_PATH)
df = clean_ohlcv(df)
df = add_intraday_features(df)
df = add_atr(df, window=14)
df = add_session_vwap(df)
df = add_continuous_session_vwap(df, session_start_hour=18)
df = compute_opening_range(df, or_minutes=baseline_cfg['or_minutes'], opening_time=baseline_cfg['opening_time'])

all_sessions = sorted(pd.to_datetime(df['session_date']).dt.date.unique())
print(f'Sessions: {all_sessions[0]} -> {all_sessions[-1]} | n={len(all_sessions)}')


ValueError: Missing required 'timestamp' or 'ts_event' column

In [ ]:
strategy = ORBStrategy(
    or_minutes=baseline_cfg['or_minutes'],
    direction=baseline_cfg['direction'],
    one_trade_per_day=True,
    entry_buffer_ticks=baseline_cfg['entry_buffer_ticks'],
    stop_buffer_ticks=baseline_cfg['stop_buffer_ticks'],
    target_multiple=baseline_cfg['target_multiple'],
    opening_time=baseline_cfg['opening_time'],
    time_exit=baseline_cfg['time_exit'],
    account_size_usd=baseline_cfg['account_size_usd'],
    risk_per_trade_pct=baseline_cfg['risk_per_trade_pct'],
    vwap_confirmation=True,
    vwap_column=baseline_cfg['vwap_column'],
)

baseline_sig = strategy.generate_signals(df)
candidate_sig = baseline_sig.copy()

signal_rows = candidate_sig['signal'].ne(0)
pass_mask = candidate_sig['atr_14'].notna()
if pd.notna(candidate_row['lower']):
    pass_mask &= candidate_sig['atr_14'] >= float(candidate_row['lower'])
if pd.notna(candidate_row['upper']):
    pass_mask &= candidate_sig['atr_14'] <= float(candidate_row['upper'])

candidate_sig.loc[signal_rows & ~pass_mask, 'signal'] = 0
candidate_sig['atr_filter_pass'] = pass_mask

engine = ExecutionModel()
baseline_trades = run_backtest(
    baseline_sig,
    execution_model=engine,
    time_exit=baseline_cfg['time_exit'],
    stop_buffer_ticks=baseline_cfg['stop_buffer_ticks'],
    target_multiple=baseline_cfg['target_multiple'],
    account_size_usd=baseline_cfg['account_size_usd'],
    risk_per_trade_pct=baseline_cfg['risk_per_trade_pct'],
    entry_on_next_open=True,
)
candidate_trades = run_backtest(
    candidate_sig,
    execution_model=engine,
    time_exit=baseline_cfg['time_exit'],
    stop_buffer_ticks=baseline_cfg['stop_buffer_ticks'],
    target_multiple=baseline_cfg['target_multiple'],
    account_size_usd=baseline_cfg['account_size_usd'],
    risk_per_trade_pct=baseline_cfg['risk_per_trade_pct'],
    entry_on_next_open=True,
)

baseline_metrics = compute_metrics(baseline_trades, signal_df=baseline_sig, session_dates=all_sessions, initial_capital=baseline_cfg['account_size_usd'])
candidate_metrics = compute_metrics(candidate_trades, signal_df=candidate_sig, session_dates=all_sessions, initial_capital=baseline_cfg['account_size_usd'])

metrics_compare = pd.DataFrame([
    {'strategy': 'baseline_exact', **baseline_metrics},
    {'strategy': candidate_name, **candidate_metrics},
])

cols = ['strategy','n_trades','profit_factor','sharpe_ratio','expectancy','max_drawdown','cumulative_pnl','time_exit_rate','time_exit_win_rate']
display(metrics_compare[cols])


In [ ]:
eq_base = build_equity_curve(baseline_trades, initial_capital=baseline_cfg['account_size_usd'])
eq_cand = build_equity_curve(candidate_trades, initial_capital=baseline_cfg['account_size_usd'])

fig, ax = plt.subplots(2, 1, figsize=(14, 9), sharex=True)
ax[0].plot(eq_base['timestamp'], eq_base['equity'], label='Baseline', color='#334155', linewidth=1.4)
ax[0].plot(eq_cand['timestamp'], eq_cand['equity'], label=candidate_name, color='#0f766e', linewidth=1.6)
ax[0].set_title('Equity Curve: Baseline vs Final Candidate')
ax[0].set_ylabel('Equity (USD)')
ax[0].legend(loc='best')

ax[1].plot(eq_base['timestamp'], eq_base['drawdown'], label='Baseline DD', color='#94a3b8', linewidth=1.2)
ax[1].plot(eq_cand['timestamp'], eq_cand['drawdown'], label=f'{candidate_name} DD', color='#b91c1c', linewidth=1.3)
ax[1].set_title('Drawdown Comparison')
ax[1].set_ylabel('Drawdown (USD)')
ax[1].legend(loc='best')

plt.tight_layout()
plt.show()


In [ ]:
def yearly_table(trades: pd.DataFrame, label: str) -> pd.DataFrame:
    out = trades.copy()
    out['year'] = pd.to_datetime(out['exit_time']).dt.year
    grouped = out.groupby('year')['net_pnl_usd']
    table = grouped.agg(n_trades='count', cumulative_pnl='sum', avg_pnl='mean').reset_index()
    table['strategy'] = label
    return table

y_base = yearly_table(baseline_trades, 'baseline_exact')
y_cand = yearly_table(candidate_trades, candidate_name)
y = pd.concat([y_base, y_cand], ignore_index=True)

display(y)

pivot = y.pivot(index='year', columns='strategy', values='cumulative_pnl').fillna(0.0)
ax = pivot.plot(kind='bar', figsize=(13,5), color=['#64748b', '#0f766e'])
ax.set_title('Yearly Cumulative PnL')
ax.set_ylabel('USD')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


In [ ]:
exit_comp = pd.concat([
    baseline_trades.assign(strategy='baseline_exact'),
    candidate_trades.assign(strategy=candidate_name),
], ignore_index=True)

reason_share = (
    exit_comp.groupby(['strategy','exit_reason']).size().rename('count').reset_index()
)
reason_share['share'] = reason_share['count'] / reason_share.groupby('strategy')['count'].transform('sum')

display(reason_share.sort_values(['strategy','share'], ascending=[True, False]))

pivot = reason_share.pivot(index='exit_reason', columns='strategy', values='share').fillna(0.0)
pivot.plot(kind='bar', figsize=(12,5), color=['#64748b', '#0f766e'])
plt.title('Exit Reason Mix (share)')
plt.ylabel('Share of trades')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


## Conclusion rapide

Ce notebook reproduit la comparaison de campagne pour le candidat final `b_atr_q20_q90`.

Points ? regarder en priorit?:
- am?lioration du PF / Sharpe / expectancy,
- r?duction du drawdown,
- baisse de participation (moins de trades) ? valider selon ton objectif de fr?quence.
